# Notebook Overview — Evaluate Development Results

## Purpose

This notebook evaluates development VideoQA results produced by the project's baseline and representation-based experiments. It discovers available experiment directories, loads valid prediction artifacts, combines them into a common evaluation dataset, attaches NExT-QA annotation metadata, computes experiment-level and question-type metrics, analyzes prediction errors, generates visual comparisons, and identifies the experiment with the highest observed accuracy.

The evaluation framework is experiment-agnostic. It can compare the Qwen2-VL baseline with any completed representation-based experiments generated by Notebook 07, including experiments using different video representation sources and fusion methods.

Evaluation is performed using a common multiple-choice framework, enabling direct comparison of all loaded experiments under identical validation, metric, error-analysis, and reporting procedures.

## Inputs

* Prediction artifacts generated by Notebooks 01 and 07
* NExT-QA annotation metadata
* Shared project configuration
* Experiment directories available in Google Drive

## Outputs

* Combined multi-experiment evaluation dataset
* Per-experiment prediction-quality and accuracy metrics
* Question-type performance metrics
* Prediction-choice distribution statistics
* Prediction error analysis
* Evaluation figures and figure inventory
* Best-experiment selection artifact
* Notebook evaluation summary

## Processing Workflow

1. Initialize the evaluation environment and discover available experiment directories.
2. Load valid prediction artifacts from all discovered experiments.
3. Attach NExT-QA annotation metadata.
4. Verify prediction quality independently for each experiment.
5. Compute experiment-level, question-type, and prediction-distribution metrics.
6. Analyze prediction errors by experiment, question type, and answer-choice pattern.
7. Generate experiment accuracy and question-type comparison figures.
8. Save evaluation artifacts and identify the experiment with the highest observed accuracy.
9. Summarize the completed multi-experiment evaluation.

## Downstream Consumer

Notebook 09 — Run Final Experiment


### 🔷 Step 1 — Initialize Evaluation Environment

* Initialize the notebook runtime and prepare the evaluation environment.
* Clone the required project repository directories and load shared configuration and metadata utilities.
* Mount Google Drive and locate the shared experiment directory.
* Discover all available experiment directories and create an experiment registry.
* Load and combine the NExT-QA annotation metadata from all dataset splits.
* Verify that required project, annotation, output, and Google Drive directories are available.
* Confirm readiness to compare available experiment prediction artifacts.




In [ ]:
# ============================================================
# Step 1: Initialize Evaluation Environment
# ============================================================

VERBOSE = True
REQUIRE_L4_GPU = False

import os
from pathlib import Path
import pandas as pd

from google.colab import userdata, drive

print("Initializing Notebook 08 environment...")
print("-" * 60)

# ------------------------------------------------------------
# Clone Required Repository Files
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):

    print("\nMounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)

else:

    print("\nGoogle Drive already mounted.")

# ------------------------------------------------------------
# Load Project Configuration
# ------------------------------------------------------------

print("\nLoading project configuration...")

from src.videoqa_representation_config import *
from src.nextqa_metadata import *

# ------------------------------------------------------------
# Resolve Experiment Root (multi-experiment aware)
# ------------------------------------------------------------

EXPERIMENTS_DRIVE_DIR = GOOGLE_DRIVE_ROOT / "experiments"

required_paths = [
    Path("src"),
    QUESTIONS_DIR,
    METADATA_DIR,
    GOOGLE_DRIVE_ROOT,
    EXPERIMENTS_DRIVE_DIR,
]

missing_paths = [
    path for path in required_paths if not Path(path).exists()
]

if missing_paths:
    for path in missing_paths:
        print(f"Missing required path: {path}")
    raise FileNotFoundError("One or more required project paths are missing.")

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print("Project paths initialized.")
print(f"Experiment root: {EXPERIMENTS_DRIVE_DIR}")

# ------------------------------------------------------------
# Discover Available Experiment Directories
# ------------------------------------------------------------

experiment_dirs = sorted(
    path for path in EXPERIMENTS_DRIVE_DIR.iterdir()
    if path.is_dir()
)

if not experiment_dirs:
    raise FileNotFoundError(
        f"No experiment directories found in: {EXPERIMENTS_DRIVE_DIR}"
    )

experiment_registry_df = pd.DataFrame({
    "experiment_name": [path.name for path in experiment_dirs],
    "experiment_dir": [str(path) for path in experiment_dirs],
})

print("\nAvailable experiment directories:")
print("-" * 60)
display(experiment_registry_df)

# ------------------------------------------------------------
# Load NExT-QA Annotation Metadata
# ------------------------------------------------------------

print("\nLoading NExT-QA annotation metadata...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

print("\nDataset metadata ready.")
print(f"Annotation records : {len(annotations_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nNotebook 08 initialization complete.")
print("-" * 60)
print("Ready to compare development experiment artifacts.")



### 🔷 Step 2 — Load Evaluation Artifacts for All Experiments

* Iterate through all experiment directories discovered during initialization.
* Infer the experiment type from each experiment name.
* Resolve the expected VideoQA prediction filename for the inferred experiment type.
* Load each available, nonempty prediction dataset.
* Skip experiment directories that do not contain a valid VideoQA prediction artifact.
* Attach the experiment name and inferred experiment type to every loaded prediction record.
* Combine all successfully loaded prediction datasets into one multi-experiment prediction table.
* Validate the common prediction schema required for downstream evaluation.
* Display the number and type of records loaded for each experiment.



In [ ]:
# ============================================================
# Step 2: Load Evaluation Artifacts for All Experiments
# ============================================================

import pandas as pd
from pathlib import Path

print("Loading ALL experiment evaluation data...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step2_objects = [
    "experiment_registry_df",
    "infer_experiment_type",
    "get_videoqa_artifact_filenames",
    "VIDEO_ID_COLUMN",
    "QUESTION_COLUMN",
]

missing_step2_objects = [
    name for name in required_step2_objects
    if name not in globals()
]

if missing_step2_objects:
    raise NameError(
        "Missing required Step 2 objects: "
        + ", ".join(missing_step2_objects)
    )

all_experiment_predictions = []

for _, row in experiment_registry_df.iterrows():

    experiment_name = row["experiment_name"]
    experiment_dir = Path(row["experiment_dir"])

    print(f"Loading experiment: {experiment_name}")

    try:
        experiment_type = infer_experiment_type(experiment_name)

        videoqa_dir = experiment_dir / "videoqa"

        if not videoqa_dir.exists():
            print(f"  ⚠ No videoqa directory: {experiment_name}")
            continue

        artifact_filenames = get_videoqa_artifact_filenames(
            experiment_type
        )

        pred_file = videoqa_dir / artifact_filenames["predictions"]

        if not pred_file.exists():
            print(f"  ⚠ Missing predictions file: {pred_file}")
            continue

        df = pd.read_csv(pred_file)

        if df.empty:
            print(f"  ⚠ Empty predictions file: {pred_file}")
            continue

        df["experiment_name"] = experiment_name
        df["experiment_type"] = experiment_type

        all_experiment_predictions.append(df)

        print(f"  ✔ Loaded {len(df):,} records")

    except Exception as e:
        print(f"  ❌ Failed {experiment_name}: {str(e)}")

# ------------------------------------------------------------
# Combine all experiments
# ------------------------------------------------------------

if not all_experiment_predictions:
    raise ValueError("No valid experiment prediction files found.")

all_predictions_df = pd.concat(
    all_experiment_predictions,
    ignore_index=True,
)

# ------------------------------------------------------------
# Validate schema
# ------------------------------------------------------------

required_cols = [
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "experiment_name",
    "experiment_type",
]

missing_cols = [
    col for col in required_cols
    if col not in all_predictions_df.columns
]

if missing_cols:
    raise ValueError(
        "Combined prediction dataset is missing required columns: "
        + ", ".join(missing_cols)
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("\nCombined Experiment Dataset")
print("-" * 60)
print(f"Total records      : {len(all_predictions_df):,}")
print(f"Experiments loaded : {all_predictions_df['experiment_name'].nunique():,}")

display(
    all_predictions_df
    .groupby(["experiment_name", "experiment_type"])
    .size()
    .reset_index(name="count")
)



### 🔷 Step 3 — Attach Annotation Metadata

* Build a compact NExT-QA annotation reference table containing video identifiers, questions, question identifiers, question types, and answer metadata when available.
* Remove duplicate annotation reference records using video identifier and question text.
* Merge the annotation reference data into the combined multi-experiment prediction dataset.
* Preserve experiment names, experiment types, ground-truth choices, predicted choices, and correctness indicators.
* Verify that the merged evaluation dataset is nonempty and contains the required evaluation columns.
* Report total records, evaluated experiment count, unique-video count, and per-experiment accuracy.
* Display representative merged evaluation records.



In [ ]:
# ============================================================
# Step 3: Attach Annotation Metadata
# ============================================================

import pandas as pd

print("Attaching annotation metadata to all experiments...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "all_predictions_df" not in globals():
    raise NameError(
        "all_predictions_df not found. Run Step 2 first."
    )

if "annotations_df" not in globals():
    raise NameError(
        "annotations_df not found. Run Step 1 first."
    )

# ------------------------------------------------------------
# Build reference table (ground truth)
# ------------------------------------------------------------

evaluation_reference_df = (
    annotations_df[
        [
            VIDEO_ID_COLUMN,
            QUESTION_COLUMN,
            "qid" if "qid" in annotations_df.columns else VIDEO_ID_COLUMN,
            "type" if "type" in annotations_df.columns else None,
            "answer" if "answer" in annotations_df.columns else None,
        ]
    ]
    .copy()
)

# Remove None columns safely
evaluation_reference_df = evaluation_reference_df[
    [c for c in evaluation_reference_df.columns if c is not None]
]

# Drop duplicates to ensure clean join
evaluation_reference_df = evaluation_reference_df.drop_duplicates(
    subset=[VIDEO_ID_COLUMN, QUESTION_COLUMN]
)

# ------------------------------------------------------------
# Merge annotations into multi-experiment predictions
# ------------------------------------------------------------

evaluation_dataset_df = all_predictions_df.merge(
    evaluation_reference_df,
    on=[VIDEO_ID_COLUMN, QUESTION_COLUMN],
    how="left"
)

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_cols = [
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "experiment_name",
]

missing_cols = [c for c in required_cols if c not in evaluation_dataset_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if evaluation_dataset_df.empty:
    raise ValueError("Evaluation dataset is empty after merge.")

# ------------------------------------------------------------
# Summary stats (IMPORTANT: grouped)
# ------------------------------------------------------------

print("Multi-Experiment Evaluation Dataset Ready")
print("-" * 60)

print(f"Total records        : {len(evaluation_dataset_df):,}")
print(f"Experiments          : {evaluation_dataset_df['experiment_name'].nunique():,}")
print(f"Unique videos        : {evaluation_dataset_df[VIDEO_ID_COLUMN].nunique():,}")

print("\nPer-experiment accuracy:")
display(
    evaluation_dataset_df.groupby("experiment_name")["choice_correct"]
    .mean()
    .reset_index(name="accuracy")
)

print("\nPreview:")
display(evaluation_dataset_df.head())



### 🔷 Step 4 — Verify Prediction Quality

* Validate prediction outputs against expected multiple-choice evaluation rules.
* Check for missing predictions, invalid predicted choices, and missing ground-truth choices.
* Confirm that predicted choices fall within the configured answer-choice set.
* Summarize prediction correctness, invalid prediction counts, and evaluation readiness.
* Display validation results before metric computation.


In [ ]:
# ============================================================
# Step 4: Verify Prediction Quality (Per-Experiment)
# ============================================================

import pandas as pd

print("Verifying prediction quality per experiment...\n")

if "evaluation_dataset_df" not in globals():
    raise NameError("evaluation_dataset_df not found. Run Step 3 first.")

required_cols = [
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
]

quality_results = []

for exp_name, df in evaluation_dataset_df.groupby("experiment_name"):

    print(f"Checking: {exp_name}")

    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"{exp_name} missing columns: {missing_cols}"
        )

    prediction_count = len(df)

    missing_ground_truth = df["ground_truth_choice"].isna().sum()
    missing_prediction = df["predicted_choice"].isna().sum()

    valid_choice_values = set(range(len(CHOICE_COLUMNS)))

    invalid_ground_truth = (
        ~df["ground_truth_choice"]
        .dropna()
        .astype(int)
        .isin(valid_choice_values)
    ).sum()

    invalid_prediction = (
        ~df["predicted_choice"]
        .dropna()
        .astype(int)
        .isin(valid_choice_values)
    ).sum()

    correct = df["choice_correct"].sum()
    incorrect = prediction_count - correct

    accuracy = correct / prediction_count if prediction_count else 0.0

    quality_results.append({
        "experiment_name": exp_name,
        "records": prediction_count,
        "missing_gt": int(missing_ground_truth),
        "missing_pred": int(missing_prediction),
        "invalid_gt": int(invalid_ground_truth),
        "invalid_pred": int(invalid_prediction),
        "correct": int(correct),
        "incorrect": int(incorrect),
        "accuracy": float(accuracy),
    })

prediction_quality_df = pd.DataFrame(quality_results)

# ------------------------------------------------------------
# Global readiness check (soft, not blocking)
# ------------------------------------------------------------

total_issues = (
    prediction_quality_df["missing_gt"].sum()
    + prediction_quality_df["missing_pred"].sum()
    + prediction_quality_df["invalid_gt"].sum()
    + prediction_quality_df["invalid_pred"].sum()
)

evaluation_ready = total_issues == 0

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nPrediction Quality Summary (Per Experiment)")
print("-" * 60)

display(prediction_quality_df)

print("\nGlobal Summary")
print("-" * 60)
print(f"Total experiments   : {len(prediction_quality_df)}")
print(f"Total records       : {prediction_quality_df['records'].sum():,}")
print(f"Evaluation ready    : {evaluation_ready}")



### 🔷 Step 5 — Expand Evaluation Metrics

* Use the per-experiment prediction-quality results from Step 4 as the primary experiment summary.
* Compute question-type prediction counts, correct-prediction counts, and accuracy independently for each experiment when question-type metadata is available.
* Compute the global distribution of predicted multiple-choice answer indices.
* Prepare experiment-summary, question-type, and prediction-distribution tables for reporting and visualization.
* Display the completed metric tables for comparison.



In [ ]:
# ============================================================
# Step 5: Expand Evaluation Metrics
# ============================================================

import pandas as pd

print("Expanding evaluation metrics...\n")

if "evaluation_dataset_df" not in globals():
    raise NameError("evaluation_dataset_df not found. Run Step 3 first.")

# ------------------------------------------------------------
# Question-type metrics (VALID per experiment + global view)
# ------------------------------------------------------------

if "type" in evaluation_dataset_df.columns:

    question_type_metrics_df = (
        evaluation_dataset_df
        .groupby(["experiment_name", "type"])
        .agg(
            prediction_count=("choice_correct", "size"),
            correct_predictions=("choice_correct", "sum"),
        )
        .reset_index()
    )

    question_type_metrics_df["choice_accuracy"] = (
        question_type_metrics_df["correct_predictions"] /
        question_type_metrics_df["prediction_count"]
    )

else:

    question_type_metrics_df = pd.DataFrame()

# ------------------------------------------------------------
# Choice distribution (still useful global diagnostic)
# ------------------------------------------------------------

choice_distribution_df = (
    evaluation_dataset_df["predicted_choice"]
    .value_counts(dropna=False)
    .rename_axis("choice")
    .reset_index(name="predicted_count")
    .sort_values("choice")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Per-experiment summary (source of truth already from Step 4)
# ------------------------------------------------------------

experiment_summary_df = prediction_quality_df.copy()

# ------------------------------------------------------------
# Display metrics
# ------------------------------------------------------------

print("Per-Experiment Summary (from Step 4)")
print("-" * 60)
display(experiment_summary_df)

print("\nQuestion-Type Metrics (Experiment-aware)")
print("-" * 60)
display(question_type_metrics_df)

print("\nPrediction Distribution (Global Diagnostic)")
print("-" * 60)
display(choice_distribution_df)



### 🔷 Step 6 — Analyze Prediction Errors

* Separate the combined evaluation dataset into correct and incorrect prediction records.
* Compute total predictions, correct predictions, incorrect predictions, and error rate independently for each experiment.
* Summarize incorrect predictions by experiment and question type when question-type metadata is available.
* Count common ground-truth versus predicted answer-choice error patterns across the combined evaluation dataset.
* Display representative incorrect predictions for qualitative inspection.
* Prepare error-analysis tables for saving and downstream interpretation.




In [ ]:
# ============================================================
# Step 6: Analyze Prediction Errors
# ============================================================

import pandas as pd

print("Analyzing prediction errors (per experiment)...\n")

if "evaluation_dataset_df" not in globals():
    raise NameError("evaluation_dataset_df was not found. Run Step 3 first.")

# ------------------------------------------------------------
# Split correct vs incorrect (global view)
# ------------------------------------------------------------

correct_predictions_df = evaluation_dataset_df[
    evaluation_dataset_df["choice_correct"] == True
].copy()

incorrect_predictions_df = evaluation_dataset_df[
    evaluation_dataset_df["choice_correct"] == False
].copy()

# ------------------------------------------------------------
# ERROR ANALYSIS BY EXPERIMENT (KEY FIX)
# ------------------------------------------------------------

error_summary_by_experiment_df = (
    evaluation_dataset_df
    .groupby("experiment_name")
    .agg(
        total=("choice_correct", "size"),
        correct=("choice_correct", "sum"),
    )
    .reset_index()
)

error_summary_by_experiment_df["incorrect"] = (
    error_summary_by_experiment_df["total"] -
    error_summary_by_experiment_df["correct"]
)

error_summary_by_experiment_df["error_rate"] = (
    error_summary_by_experiment_df["incorrect"] /
    error_summary_by_experiment_df["total"]
)

# ------------------------------------------------------------
# ERROR TYPE BY EXPERIMENT (VERY IMPORTANT)
# ------------------------------------------------------------

if "type" in evaluation_dataset_df.columns:

    error_type_by_experiment_df = (
        incorrect_predictions_df
        .groupby(["experiment_name", "type"])
        .size()
        .reset_index(name="error_count")
        .sort_values(["experiment_name", "error_count"], ascending=[True, False])
    )

else:
    error_type_by_experiment_df = pd.DataFrame()

# ------------------------------------------------------------
# ERROR PATTERNS (global but still useful)
# ------------------------------------------------------------

error_choice_pattern_df = (
    incorrect_predictions_df
    .groupby(["ground_truth_choice", "predicted_choice"])
    .size()
    .reset_index(name="error_count")
    .sort_values("error_count", ascending=False)
)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("Error Summary by Experiment")
print("-" * 60)
display(error_summary_by_experiment_df)

print("\nError Types by Experiment")
print("-" * 60)
display(error_type_by_experiment_df)

print("\nGlobal Error Patterns (Confusions)")
print("-" * 60)
display(error_choice_pattern_df)

print("\nExample Errors")
print("-" * 60)

if incorrect_predictions_df.empty:
    print("No errors found.")
else:
    display(incorrect_predictions_df.head(10))



### 🔷 Step 7 — Generate Evaluation Visualizations

* Create a bar chart comparing overall multiple-choice accuracy across all evaluated experiments.
* Create a question-type accuracy comparison chart when question-type metrics are available.
* Save the generated figures to the local evaluation figures directory.
* Create and save a figure-inventory table containing the name and path of each generated visualization.
* Display the completed figure inventory.




In [ ]:
# ============================================================
# Step 7: Generate Evaluation Visualizations
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd

print("Generating multi-experiment evaluation visualizations...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "prediction_quality_df" not in globals():
    raise NameError(
        "prediction_quality_df was not found. Run Step 4 first."
    )

if "question_type_metrics_df" not in globals():
    raise NameError(
        "question_type_metrics_df was not found. Run Step 5 first."
    )

# ------------------------------------------------------------
# Create output directories (Notebook 08 global outputs)
# ------------------------------------------------------------

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

evaluation_output_root = OUTPUTS_DIR / "evaluation"
evaluation_output_root.mkdir(parents=True, exist_ok=True)

evaluation_figures_dir = evaluation_output_root / "figures"
evaluation_figures_dir.mkdir(parents=True, exist_ok=True)

generated_figures = []

# ============================================================
# 1. EXPERIMENT ACCURACY COMPARISON (PRIMARY METRIC)
# ============================================================

plt.figure(figsize=(6, 4))

plt.bar(
    prediction_quality_df["experiment_name"],
    prediction_quality_df["accuracy"],
)

plt.title("Experiment Accuracy Comparison")
plt.xlabel("Experiment")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20)

plt.tight_layout()

fig1 = evaluation_figures_dir / "experiment_accuracy.png"
plt.savefig(fig1, dpi=150)
plt.show()

generated_figures.append({
    "figure": "experiment_accuracy",
    "path": str(fig1),
})

# ============================================================
# 2. QUESTION TYPE ACCURACY BY EXPERIMENT
# ============================================================

if not question_type_metrics_df.empty:

    plt.figure(figsize=(10, 5))

    for exp in question_type_metrics_df["experiment_name"].unique():

        subset = question_type_metrics_df[
            question_type_metrics_df["experiment_name"] == exp
        ]

        plt.plot(
            subset["type"].astype(str),
            subset["choice_accuracy"],
            marker="o",
            label=exp,
        )

    plt.title("Question Type Accuracy by Experiment")
    plt.xlabel("Question Type")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()

    fig2 = evaluation_figures_dir / "question_type_by_experiment.png"
    plt.savefig(fig2, dpi=150)
    plt.show()

    generated_figures.append({
        "figure": "question_type_by_experiment",
        "path": str(fig2),
    })

# ============================================================
# SAVE FIGURE INVENTORY
# ============================================================

generated_figures_df = pd.DataFrame(generated_figures)

fig_inventory = evaluation_figures_dir / "generated_figures.csv"
generated_figures_df.to_csv(fig_inventory, index=False)

print("Evaluation visualizations generated.")
print("-" * 60)
print(f"Figures created: {len(generated_figures_df)}")

display(generated_figures_df)



### 🔷 Step 8 — Save Evaluation Artifacts and Final Experiment Selection

* Verify that the required evaluation, metric, error-analysis, and figure-inventory datasets are available.
* Create local and Google Drive evaluation output directories.
* Save the combined evaluation dataset.
* Save per-experiment prediction-quality metrics.
* Save question-type metrics, prediction-choice distribution data, answer-choice error patterns, and the generated-figure inventory.
* Select the experiment with the highest observed accuracy.
* Create `best_model.json` containing the selected experiment, its accuracy, and the results for all evaluated experiments.
* Copy the best-experiment selection artifact to the Google Drive evaluation directory.
* Display the selected experiment, its accuracy, and the local evaluation output location.





In [ ]:
# ============================================================
# Step 8: Save Evaluation Artifacts + Final Experiment Selection
# ============================================================

import pandas as pd
import shutil
import json
from pathlib import Path

print("Saving evaluation results...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_dataframes = {
    "evaluation_dataset_df": "Step 3",
    "prediction_quality_df": "Step 4",
    "question_type_metrics_df": "Step 5",
    "choice_distribution_df": "Step 5",
    "error_type_by_experiment_df": "Step 6" if "error_type_by_experiment_df" in globals() else "Step 6",
    "error_choice_pattern_df": "Step 6",
    "generated_figures_df": "Step 7",
}

missing_dataframes = [
    name for name in required_dataframes
    if name not in globals()
]

if missing_dataframes:
    raise NameError(f"Missing dataframes: {missing_dataframes}")

# ============================================================
# FIX: Notebook 08 evaluation output directory (STRICT MODE SAFE)
# ============================================================

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

EVALUATION_OUTPUT_DIR = OUTPUTS_DIR / "evaluation"
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVALUATION_OUTPUT_DRIVE_DIR = (
    GOOGLE_DRIVE_ROOT / "evaluation"
)
EVALUATION_OUTPUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Define outputs
# ------------------------------------------------------------

outputs = [
    (evaluation_dataset_df, EVALUATION_OUTPUT_DIR / "evaluation_dataset.csv"),
    (prediction_quality_df, EVALUATION_OUTPUT_DIR / "prediction_quality.csv"),
    (question_type_metrics_df, EVALUATION_OUTPUT_DIR / "question_type_metrics.csv"),
    (choice_distribution_df, EVALUATION_OUTPUT_DIR / "choice_distribution.csv"),
    (error_choice_pattern_df, EVALUATION_OUTPUT_DIR / "error_choice_pattern.csv"),
    (generated_figures_df, EVALUATION_OUTPUT_DIR / "generated_figures.csv"),
]

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

for df, path in outputs:

    df.to_csv(path, index=False)

    if not path.exists():
        raise FileNotFoundError(f"Failed to save: {path}")

# ------------------------------------------------------------
# CREATE FINAL MODEL SELECTION ARTIFACT (IMPORTANT)
# ------------------------------------------------------------

best_experiment_row = (
    prediction_quality_df
    .sort_values("accuracy", ascending=False)
    .iloc[0]
)

best_model_artifact = {
    "best_experiment": best_experiment_row["experiment_name"],
    "best_accuracy": float(best_experiment_row["accuracy"]),
    "all_experiments": prediction_quality_df.to_dict(orient="records"),
}

best_model_path = EVALUATION_OUTPUT_DIR / "best_model.json"

with open(best_model_path, "w") as f:
    json.dump(best_model_artifact, f, indent=4)

# ------------------------------------------------------------
# PROMOTE TO GOOGLE DRIVE
# ------------------------------------------------------------

EVALUATION_OUTPUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for _, row in prediction_quality_df.iterrows():
    pass  # (optional extension point)

shutil.copy2(best_model_path, EVALUATION_OUTPUT_DRIVE_DIR / best_model_path.name)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("Evaluation results saved.")
print("-" * 60)
print(f"Best experiment : {best_model_artifact['best_experiment']}")
print(f"Best accuracy   : {best_model_artifact['best_accuracy']:.3f}")
print(f"Output dir      : {EVALUATION_OUTPUT_DIR}")



### 🔷 Step 9 — Notebook Summary

* Summarize the completed multi-experiment development evaluation.
* Report the total prediction count and number of experiments evaluated.
* Display per-experiment records, correct predictions, incorrect predictions, and accuracy.
* Display question-type performance and available error-analysis results.
* Report the location of the best-experiment selection artifact.
* Generate a text summary containing the overall experiment count and per-experiment prediction-quality table.
* Save the text summary locally and copy it to the Google Drive evaluation directory.
* Confirm readiness for the planned final-experiment workflow.




In [ ]:
# ============================================================
# Step 9: Notebook Summary
# ============================================================

print("Notebook 08 complete.")
print("=" * 60)

# ------------------------------------------------------------
# Verify REQUIRED objects only (no optional dependencies)
# ------------------------------------------------------------

required_objects = {
    "evaluation_dataset_df": "Step 3",
    "prediction_quality_df": "Step 4",
    "question_type_metrics_df": "Step 5",
}

missing_required = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_required:
    raise NameError(
        f"Missing REQUIRED objects: {missing_required}"
    )

# ------------------------------------------------------------
# Safe optional objects (do not break notebook)
# ------------------------------------------------------------

optional_objects = [
    "incorrect_predictions_df",
    "error_type_by_experiment_df",
]

for obj in optional_objects:
    if obj not in globals():
        globals()[obj] = None

# ------------------------------------------------------------
# CORE SUMMARY METRICS (MULTI-EXPERIMENT SAFE)
# ------------------------------------------------------------

total_predictions = len(evaluation_dataset_df)
correct_predictions = int(evaluation_dataset_df["choice_correct"].sum())
incorrect_predictions = total_predictions - correct_predictions

print("\nMulti-Experiment Evaluation Summary")
print("-" * 60)

print(f"Total predictions        : {total_predictions:,}")
print(f"Experiments evaluated    : {evaluation_dataset_df['experiment_name'].nunique():,}")

# ------------------------------------------------------------
# PER-EXPERIMENT RESULTS (PRIMARY OUTPUT)
# ------------------------------------------------------------

print("\nPer-Experiment Results")
print("-" * 60)

display(
    prediction_quality_df[[
        "experiment_name",
        "records",
        "correct",
        "incorrect",
        "accuracy"
    ]]
)

# ------------------------------------------------------------
# QUESTION TYPE PERFORMANCE
# ------------------------------------------------------------

print("\nQuestion-Type Performance")
print("-" * 60)

if question_type_metrics_df is not None and not question_type_metrics_df.empty:
    display(question_type_metrics_df)
else:
    print("No question-type metrics available.")

# ------------------------------------------------------------
# ERROR SUMMARY (ONLY IF AVAILABLE)
# ------------------------------------------------------------

print("\nError Analysis Summary")
print("-" * 60)

if incorrect_predictions_df is not None:
    print(f"Incorrect predictions: {len(incorrect_predictions_df):,}")
else:
    print("Error analysis not available (optional Step 6 outputs missing).")

if error_type_by_experiment_df is not None:
    print("\nErrors by Experiment + Type")
    display(error_type_by_experiment_df)

# ------------------------------------------------------------
# FINAL MODEL SELECTION OUTPUT
# ------------------------------------------------------------

print("\nBest Model Selection (from Step 8)")
print("-" * 60)

if "best_model_path" in globals():
    print(f"Best model artifact saved at: {best_model_path}")
else:
    print("Best model artifact not available in this run.")

summary_path = EVALUATION_OUTPUT_DIR / "notebook08_summary.txt"

with open(summary_path, "w") as f:
    f.write("Notebook 08 Complete\n")
    f.write("=" * 40 + "\n")
    f.write(f"Total predictions: {total_predictions}\n")
    f.write(f"Experiments: {evaluation_dataset_df['experiment_name'].nunique()}\n")
    f.write("\nPer-experiment accuracy:\n")
    f.write(prediction_quality_df.to_string(index=False))

shutil.copy2(
    summary_path,
    EVALUATION_OUTPUT_DRIVE_DIR / summary_path.name
)

# ------------------------------------------------------------
# FINAL SYSTEM MESSAGE
# ------------------------------------------------------------

print("\nNotebook 08 Summary")
print("-" * 60)

print("✔ Multi-experiment evaluation completed")
print("✔ Representation-based pipelines evaluated")
print("✔ Baseline pipeline evaluated")
print("✔ Best experiment identified")
print("✔ Notebook 08 pipeline finished")

print("\nReady for Notebook 09 (Final Full Dataset Evaluation)")

